# Questão 4
## 1. Motivação
Há diferença no financiamento de campanha entre candidatos e candidatas?

As questões 1 e 2 mostraram que a cota encheu as listas de mulheres (cerca de um terço das candidaturas proporcionais desde 2014), mas que a fração de eleitas cresceu bem menos e que uma candidata tem cerca de 0,3 a 0,4 da chance de um candidato de se eleger. Uma explicação comum para essa diferença é o dinheiro: a lei obriga o partido a lançar candidatas, mas, até 2018, não o obrigava a financiá-las.

Isso mudou em 2018. Em março, o STF (ADI 5617) determinou que pelo menos 30% dos recursos do Fundo Partidário usados em campanhas fossem para candidatas, na proporção das candidaturas femininas, e em maio o TSE estendeu a regra ao Fundo Especial de Financiamento de Campanha (FEFC, o "Fundo Eleitoral"), criado em 2017. Em 2022, a Emenda Constitucional 117 passou a regra para a Constituição. Outro marco é a proibição de doações de empresas, decidida pelo STF em 2015 e válida a partir de 2016. Por isso, analisamos de 2014 a 2024 e marcamos 2018 como a primeira eleição com o piso de 30% para o dinheiro público.

Traduzimos a pergunta em três perguntas de dados:

1. **Participação:** que fração do dinheiro de campanha (total e do FEFC) vai para mulheres, comparada à fração de candidaturas femininas?
2. **Distribuição:** quanto arrecada uma candidata típica em relação a um candidato típico, e quantas candidaturas de cada gênero não declaram nenhuma receita?
3. **Dinheiro e resultado:** comparando candidatas e candidatos com arrecadação parecida (mesma faixa dentro da sua disputa), a diferença na chance de eleição diminui?

## 2. Hipótese
Esperamos que, antes de 2018, a fração de recursos recebida por mulheres fique abaixo da fração de candidaturas femininas, e que a partir de 2018 a fração do FEFC destinada a elas se aproxime de 30%, sem que a fração do total chegue à das candidaturas, porque doações privadas e recursos próprios não estão sujeitos à regra. Esperamos também que a mediana de arrecadação das candidatas seja menor que a dos candidatos e que mais candidatas tenham receita zero. Por fim, esperamos que, dentro de uma mesma faixa de arrecadação, a taxa de eleição de mulheres e homens fique mais próxima, de modo que parte da diferença vista na questão 2 esteja associada à distribuição do dinheiro.

A hipótese é refutada se a fração de recursos das mulheres for igual ou maior que a de candidaturas, ou se a diferença na taxa de eleição continuar do mesmo tamanho dentro de cada faixa de arrecadação.

## 3. Dados

### 3.1 Candidaturas e resultados (mesmo recorte das questões 1 a 3)

In [1]:
import unicodedata

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook"
pio.templates.default = "plotly_white"

COR_GENERO = {"Feminino": "#B5338A", "Masculino": "#2B7BB9"}   # as mesmas cores da questão 2

df = pd.read_parquet("../data/resultados_candidato.parquet")

# eleito: inclui eleito, eleito por qp e eleito por media (suplentes não contam)
df["eleito"] = df["resultado"].fillna("").str.startswith("eleito")

# senador MT 2018: Selma Arruda (PSL), eleita e cassada depois, aparece sem resultado (ver questão 2)
mt18 = ((df["cargo"] == "senador") & (df["ano"] == 2018) &
        (df["sigla_uf"] == "MT") & (df["tipo_eleicao"] == "eleicao ordinaria"))
selma = mt18 & (df["sigla_partido"] == "PSL") & (df["genero"] == "feminino")
df.loc[selma, "eleito"] = True

base = df[(df["tipo_eleicao"] == "eleicao ordinaria") & (df["ano"] >= 2014)].copy()
base = base[base["genero"].isin(["feminino", "masculino"])]

# deputado distrital entra junto com estadual (mesma decisão da questão 1);
# presidente fica de fora (de 11 a 14 candidaturas por eleição)
base["cargo"] = base["cargo"].replace({"deputado distrital": "deputado estadual"})
CARGOS = ["deputado federal", "deputado estadual", "vereador", "senador", "governador", "prefeito"]
base = base[base["cargo"].isin(CARGOS)]

invalidas = ["indeferido", "indeferido com recurso", "renuncia", "cancelado",
             "cancelado com recurso", "cassado", "cassado com recurso",
             "falecido", "falecimento", "inelegivel", "impugnado",
             "nao conhecimento do pedido", "pedido nao conhecido",
             "pedido nao conhecido com recurso"]
base["valida"] = ~base["situacao"].isin(invalidas) | base["eleito"]

base["sequencial"] = base["sequencial"].astype(str).str.strip()
base["Gênero"] = base["genero"].map({"feminino": "Feminino", "masculino": "Masculino"})

print(base.shape)
print("chaves (ano, sequencial) repetidas:", base.duplicated(["ano", "sequencial"]).sum())

(1544035, 15)
chaves (ano, sequencial) repetidas: 0


### 3.2 Receitas de campanha

As receitas vêm da tabela `br_tse_eleicoes.receitas_candidato` (prestação de contas ao TSE), já agregadas no BigQuery por candidatura, fonte, origem e natureza da receita (consulta em `queries/receitas_candidato.sql`, baixada em `00_download.ipynb`). Os rótulos de texto da Base dos Dados variam entre anos, então normalizamos (minúsculas, sem acentos) e conferimos os valores antes de classificar.

In [2]:
rec = pd.read_parquet("../data/receitas_candidato.parquet")


def normaliza(s):
    # minúsculas, sem acento; aplicado aos valores únicos, para ser rápido
    s = s.fillna("nao informado").astype(str)
    mapa = {v: unicodedata.normalize("NFKD", v).encode("ascii", "ignore").decode().lower().strip()
            for v in s.unique()}
    return s.map(mapa)


for c in ["fonte_receita", "origem_receita", "natureza_receita"]:
    rec[c] = normaliza(rec[c])

rec["valor"] = rec["valor"].astype(float)
rec["sequencial"] = rec["sequencial_candidato"].astype(str).str.strip()
print(rec.shape)

# quanto dinheiro (R$ milhões) há em cada rótulo, por ano: confira antes de seguir
for c in ["fonte_receita", "origem_receita", "natureza_receita"]:
    display((rec.pivot_table(index=c, columns="ano", values="valor", aggfunc="sum", fill_value=0) / 1e6).round(1))

(3189360, 12)


ano,2014,2016,2018,2020,2022,2024
fonte_receita,,,,,,
fundo especial,0.0,0.0,1833.7,1915.8,4899.9,6142.9
fundo partidario,90.1,352.2,425.9,149.0,494.7,219.0
nao especificado,2166.0,0.0,0.0,0.0,0.0,0.0
nao informado,0.0,0.0,0.0,0.0,0.0,0.0
outros recursos,4.2,2649.7,1079.9,4258.8,1235.8,1823.8
outros recursos nao descritos,2136.6,0.0,0.0,0.0,0.0,0.0


ano,2014,2016,2018,2020,2022,2024
origem_receita,,,,,,
comercializacao de bens com fefc,0.0,0.0,0.0,0.0,0.0,0.0
comercializacao de bens com or,0.0,0.0,0.0,0.0,0.0,0.0
comercializacao de bens ou realizacao de eventos,0.0,0.1,0.1,0.1,0.0,0.0
doacoes pela internet,1.2,2.5,0.3,0.3,0.1,6.5
fundo partidario,0.0,0.0,0.0,0.0,0.0,0.0
nao informado,0.0,0.0,0.0,0.0,0.0,0.0
recursos de financiamento coletivo,0.0,0.0,19.3,15.9,21.2,8.1
recursos de origens nao identificadas,0.6,0.5,0.8,0.7,0.3,0.8
recursos de outros candidatos,0.0,149.6,203.5,183.3,197.2,251.3


ano,2014,2016,2018,2020,2022,2024
natureza_receita,,,,,,
--,0.2,0.0,0.0,0.0,0.0,0.0
boleto de cobranca,0.0,0.5,0.0,0.0,0.0,0.0
cartao de credito,0.8,1.8,0.0,0.0,0.0,0.0
cartao de debito,0.0,0.2,0.0,0.0,0.0,0.0
cheque,996.3,82.2,0.0,0.0,0.0,0.0
deposito em especie,373.9,370.8,0.0,0.0,0.0,0.0
em especie,6.0,14.8,0.0,0.0,0.0,0.0
estimado,958.9,934.6,0.0,0.0,0.0,0.0
estimavel,0.0,0.0,280.5,3116.8,384.4,611.6


In [3]:
def classe_fonte(f):
    if "fundo especial" in f or "fefc" in f:
        return "FEFC"
    if "fundo partidario" in f:
        return "Fundo Partidário"
    return "Outros recursos"


def classe_origem(o):
    # a ordem importa: "recursos de outros candidatos" contém "candidato"
    if "propri" in o:
        return "Recursos próprios"
    if "outros candidatos" in o:
        return "Outros candidatos"
    if "partido" in o:
        return "Partido"
    if "pessoas fisicas" in o:
        return "Pessoas físicas"
    if "pessoas juridicas" in o:
        return "Empresas"
    if "coletivo" in o or "internet" in o:
        return "Financiamento coletivo"
    return "Outras origens"


rec["fonte"] = rec["fonte_receita"].map({v: classe_fonte(v) for v in rec["fonte_receita"].unique()})
rec["origem"] = rec["origem_receita"].map({v: classe_origem(v) for v in rec["origem_receita"].unique()})
# 2018 em diante: "estimavel"; 2014 e 2016: a coluna traz a forma de pagamento, e bens e serviços são "estimado"
rec["estimavel"] = rec["natureza_receita"].str.contains("estimav") | (rec["natureza_receita"] == "estimado")

# confere a classificação: cada rótulo original caiu na classe certa?
display(rec.groupby(["fonte", "fonte_receita"])["valor"].sum().div(1e6).round(1).to_frame("R$ mi"))
display(rec.groupby(["origem", "origem_receita"])["valor"].sum().div(1e6).round(1).to_frame("R$ mi"))

R$ mi
fonte            fonte_receita                         
FEFC             fundo especial                 14792.4
Fundo Partidário fundo partidario                1730.8
Outros recursos  nao especificado                2166.0
                 nao informado                      0.0
                 outros recursos                11052.1
                 outros recursos nao descritos   2136.6

R$ mi
origem                 origem_receita                                           
Empresas               recursos de pessoas juridicas                      1299.4
Financiamento coletivo doacoes pela internet                                10.8
                       recursos de financiamento coletivo                   64.5
Outras origens         comercializacao de bens com fefc                      0.0
                       comercializacao de bens com or                        0.1
                       comercializacao de bens ou realizacao de eventos      0.3
                       fundo partidario                                      0.0
                       nao informado                                         0.0
                       recursos de origens nao identificadas                 3.7
                       rendimentos de aplicacoes financeiras                 1.6
Outros candidatos      recursos de outros candidatos                       984.9
                       recursos de outros candidatos/comites               918.9
Partido                recursos de partido politico                      20082.4
Pessoas físicas        recursos de pessoas fisicas                        5446.0
Recursos próprios      recursos proprios                                  3065.3

In [4]:
# uma linha por candidatura
por_cand = rec.groupby(["ano", "sequencial"]).agg(
    receita_total=("valor", "sum"),
    n_receitas=("n_receitas", "sum"),
)
por_cand["receita_financeira"] = rec[~rec["estimavel"]].groupby(["ano", "sequencial"])["valor"].sum()
por_cand["fefc"] = rec[rec["fonte"] == "FEFC"].groupby(["ano", "sequencial"])["valor"].sum()
por_cand["fundo_partidario"] = rec[rec["fonte"] == "Fundo Partidário"].groupby(["ano", "sequencial"])["valor"].sum()
origens = rec.pivot_table(index=["ano", "sequencial"], columns="origem", values="valor", aggfunc="sum")
por_cand = por_cand.join(origens).fillna(0).reset_index()

cand = base.merge(por_cand, on=["ano", "sequencial"], how="left", validate="one_to_one")
cand["tem_receita"] = cand["n_receitas"].notna() & (cand["receita_total"] > 0)
cols_valor = [c for c in por_cand.columns if c not in ["ano", "sequencial"]]
cand[cols_valor] = cand[cols_valor].fillna(0)

# cobertura do cruzamento: quanto do dinheiro declarado encontrou a sua candidatura
casou = rec.merge(base[["ano", "sequencial"]], on=["ano", "sequencial"], how="left", indicator=True)
cob = casou.groupby("ano").apply(
    lambda d: pd.Series({
        "R$ mi declarados": d["valor"].sum() / 1e6,
        "% do valor cruzado": d.loc[d["_merge"] == "both", "valor"].sum() / d["valor"].sum() * 100,
    }), include_groups=False)
cob.round(1)

,R$ mi declarados,% do valor cruzado
ano,,
2014,4396.9,85.2
2016,3001.9,99.3
2018,3339.4,93.3
2020,6323.7,99.3
2022,6630.4,94.2
2024,8185.6,100.0


In [5]:
# o que ficou sem cruzar? (esperado: suplementares, presidente, vices e suplentes de senador)
sem = casou[casou["_merge"] == "left_only"]
(sem.groupby(["tipo_eleicao", "cargo"])["valor"].sum().div(1e6).sort_values(ascending=False).head(10).round(1)
    .to_frame("R$ mi sem cruzar"))

R$ mi sem cruzar
tipo_eleicao                         cargo                              
eleicao ordinaria                    presidente                   1247.9
eleicoes municipais 2020 - ap        vereador                       11.8
                                     prefeito                        7.5
eleicao suplementar governador am    governador                      5.4
eleicao ordinaria                    deputado federal                4.9
                                     deputado estadual               1.8
                                     vice-prefeito                   1.6
eleicao suplementar brusque set/2023 prefeito                        1.5
eleicao suplementar de coari/am      prefeito                        1.4
eleicao suplementar ipojuca pe       prefeito                        1.3

### 3.3 Decisões

- **Período:** 2014 a 2024. Antes de 2014 a prestação de contas é menos completa (ver proposta). O FEFC só existe a partir de 2018.
- **Universo:** candidaturas válidas (mesma definição da questão 2): candidaturas indeferidas, renunciadas ou canceladas costumam ter pouca ou nenhuma receita e inflariam a fração de receitas zero.
- **Receita da candidatura:** soma de todas as receitas declaradas pela candidatura, inclusive as estimáveis em dinheiro (bens e serviços doados). Repasses do partido e de outros candidatos contam como receita de quem recebe, porque é dinheiro disponível para aquela campanha. Na análise de robustez usamos só a receita financeira.
- **Valores nominais:** não corrigimos pela inflação. Por isso só comparamos valores em reais dentro de um mesmo ano; entre anos, comparamos frações e razões, que não dependem da inflação.
- **Receita zero:** a candidatura válida que não aparece na tabela de receitas é tratada como receita zero. Isso mistura quem de fato não arrecadou com quem não prestou contas; a fração aparece separada na seção 5.
- **Cargos:** deputado distrital somado a estadual (como na questão 1); presidente fora, por ter poucas candidaturas.

## 4. Participação das mulheres no dinheiro de campanha

In [6]:
v = cand[cand["valida"]].copy()


def participacao(d, chaves):
    t = d.groupby(chaves + ["genero"], observed=True).agg(
        n=("sequencial", "size"), receita=("receita_total", "sum"), fefc=("fefc", "sum"))
    t = t.unstack("genero", fill_value=0)
    tot = lambda m: t[(m, "feminino")] + t[(m, "masculino")]
    out = pd.DataFrame({
        "n_cand": tot("n"),
        "pct_cand": t[("n", "feminino")] / tot("n") * 100,
        "receita_mi": tot("receita") / 1e6,
        "pct_receita": t[("receita", "feminino")] / tot("receita") * 100,
        "fefc_mi": tot("fefc") / 1e6,
        "pct_fefc": t[("fefc", "feminino")] / tot("fefc").replace(0, np.nan) * 100,
    })
    out["razao_receita_cand"] = out["pct_receita"] / out["pct_cand"]
    return out.reset_index()


part = participacao(v, ["ano", "cargo"])
part.round(1)

,ano,cargo,n_cand,pct_cand,receita_mi,pct_receita,fefc_mi,pct_fefc,razao_receita_cand
0,2014,deputado estadual,15250,29.1,1249.6,13.1,0.0,NaN,0.5
1,2014,deputado federal,5868,29.4,1162.9,9.8,0.0,NaN,0.3
2,2014,governador,162,11.7,1021.8,4.6,0.0,NaN,0.4
3,2014,senador,164,20.7,271.1,16.8,0.0,NaN,0.8
4,2016,prefeito,15578,13.1,1445.9,11.8,0.0,NaN,0.9
5,2016,vereador,435741,32.5,1427.2,21.1,0.0,NaN,0.7
6,2018,deputado estadual,16941,31.2,1030.4,30.1,414.2,48.1,1.0
7,2018,deputado federal,7629,31.7,1330.1,22.3,830.4,24.9,0.7
8,2018,governador,183,15.3,408.9,9.6,243.4,13.5,0.6
9,2018,senador,309,17.8,287.2,16.3,180.5,20.4,0.9


In [7]:
SERIES = {"pct_cand": "Das candidaturas",
          "pct_receita": "Do dinheiro total",
          "pct_fefc": "Do Fundo Eleitoral (FEFC)"}
nomes = list(SERIES.values())
COR_SERIE = dict(zip(nomes, ["#7F7F7F", "#B5338A", "#6E1F54"]))
DASH_SERIE = dict(zip(nomes, ["dot", "solid", "dash"]))
SIMB_SERIE = dict(zip(nomes, ["circle-open", "diamond", "square"]))

longo = part.melt(id_vars=["ano", "cargo", "n_cand", "receita_mi", "fefc_mi"],
                  value_vars=list(SERIES), var_name="serie", value_name="pct").dropna(subset=["pct"])
longo["serie"] = longo["serie"].map(SERIES)
longo["cargo"] = longo["cargo"].str.title()
longo["ano"] = longo["ano"].astype(int)
longo["pct"] = longo["pct"].astype(float)

# título com o achado, calculado a partir dos dados
ref = part[(part["cargo"] == "deputado federal")].set_index("ano")
a0, a1 = ref.index.min(), ref.index.max()
titulo = (f"Deputado federal: as mulheres receberam {ref.loc[a0, 'pct_receita']:.0f}% dos recursos em {a0} "
          f"e {ref.loc[a1, 'pct_receita']:.0f}% em {a1}, para {ref.loc[a1, 'pct_cand']:.0f}% das candidaturas"
          "<br><sup>% das candidaturas válidas e % do dinheiro recebido que ficaram com mulheres. "
          "O FEFC existe desde 2018; desde então, ao menos 30% dele deve ir para candidatas.</sup>")

fig1 = px.line(
    longo, x="ano", y="pct", color="serie", line_dash="serie", symbol="serie", markers=True,
    facet_col="cargo", facet_col_wrap=3,
    category_orders={"cargo": [c.title() for c in CARGOS], "serie": list(SERIES.values())},
    color_discrete_map=COR_SERIE, line_dash_map=DASH_SERIE, symbol_map=SIMB_SERIE,
    hover_data={"n_cand": ":,", "receita_mi": ":,.1f", "fefc_mi": ":,.1f", "pct": ":.1f",
                "cargo": False, "serie": False},
    labels={"ano": "Ano da eleição", "pct": "% que ficou com mulheres", "serie": "",
            "n_cand": "Candidaturas válidas", "receita_mi": "Recursos (R$ mi)", "fefc_mi": "FEFC (R$ mi)"},
    title=titulo,
)
fig1.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig1.update_xaxes(matches=None, showticklabels=True, type="category", categoryorder="category ascending")
fig1.update_yaxes(range=[0, max(50, np.ceil(longo["pct"].max() / 10) * 10)], ticksuffix="%")
fig1.add_hline(y=30, line_dash="dot", line_color="#444", line_width=1,
               annotation_text="30%", annotation_position="bottom left", annotation_font_size=10,
               row="all", col="all")
fig1.update_layout(height=680, margin=dict(t=120), legend=dict(orientation="h", y=1.08, x=1, xanchor="right"),
                   title_font_size=16)
fig1.show()

In [8]:
# razão entre a fração de recursos e a fração de candidaturas (1 = proporcional)
part.pivot(index="ano", columns="cargo", values="razao_receita_cand")[CARGOS].round(2)

cargo,deputado federal,deputado estadual,vereador,senador,governador,prefeito
ano,,,,,,
2014,0.33,0.45,NaN,0.81,0.39,NaN
2016,NaN,NaN,0.65,NaN,NaN,0.90
2018,0.70,0.97,NaN,0.92,0.63,NaN
2020,NaN,NaN,1.09,NaN,NaN,1.34
2022,0.85,0.98,NaN,0.95,0.58,NaN
2024,NaN,NaN,0.71,NaN,NaN,1.16


In [9]:
# fração do FEFC que ficou com mulheres, somando os cargos analisados de cada eleição
# (sem presidente, vices e suplentes; não verifica o piso de 30%, que é calculado por partido)
f = v.groupby(["ano", "genero"])["fefc"].sum().unstack()
(f["feminino"] / f.sum(axis=1) * 100).dropna().round(1).to_frame("% do FEFC para mulheres")

,% do FEFC para mulheres
ano,
2018,28.5
2020,30.2
2022,31.1
2024,22.3


### Análise: participação

Antes de 2018, as mulheres receberam uma fração do dinheiro bem menor que a fração de candidaturas. Em 2014, as candidatas a deputado federal eram 29,4% das candidaturas e receberam 9,8% dos recursos (razão 0,33): para cada real que receberiam se o dinheiro fosse dividido na proporção das candidaturas, receberam cerca de 33 centavos. Para deputado estadual, a razão era 0,45 (29,1% das candidaturas e 13,1% dos recursos), e para vereador, em 2016, 0,65 (32,5% e 21,1%).

A partir de 2018, a distância diminui nos cargos proporcionais. Para deputado federal, a fração de recursos das mulheres sobe para 22,3% em 2018 e 30,0% em 2022, para 31,7% e 35,3% das candidaturas (razões 0,70 e 0,85). Para deputado estadual, as duas linhas praticamente se encontram (razões 0,97 e 0,98). Para vereador, a razão vai de 0,65 em 2016 para 1,09 em 2020 e volta para 0,71 em 2024. O valor de 2020 depende das receitas estimáveis (bens e serviços), que somaram 3,1 bilhões de reais naquele ano, quase metade de tudo o que foi declarado: contando só a receita financeira, a fração das vereadoras cai de 37,9% para 32,8%, abaixo dos 34,7% de candidaturas (seção 7).

Nos cargos majoritários, o padrão varia. Para governador, a razão fica entre 0,39 e 0,63 em todas as eleições. Para senador, fica perto de 1 (de 0,81 a 0,95). Para prefeito, passa de 0,90 em 2016 para 1,34 em 2020 e 1,16 em 2024: as poucas candidatas a prefeita (13% a 15% do total) recebem uma fração do dinheiro maior que a presença delas na disputa.

A fração do Fundo Eleitoral que ficou com mulheres varia muito entre cargos: em 2022, 39,6% para deputado estadual, 32,0% para deputado federal, 25,5% para senador e 11,1% para governador; em 2024, 25,1% para vereador e 19,3% para prefeito. O piso de 30% vale para o total que o partido destina às campanhas, e não para cada cargo, então uma fração abaixo de 30% em um cargo não indica, sozinha, descumprimento. A célula abaixo soma os cargos analisados em cada eleição. Mesmo assim, ela não verifica a regra, porque deixa de fora vices e suplentes (recursos para chapas com vice mulher também contam para o piso) e não separa por partido. Para deputado federal, a fração do FEFC (32,0% em 2022) fica pouco acima da fração do total (30,0%), o que é compatível com a ideia de que o dinheiro fora da regra vai menos para mulheres, mas a diferença é pequena.

Em relação à hipótese: a primeira parte se confirma (antes de 2018, as mulheres recebiam muito menos que a sua fração de candidaturas). A segunda se confirma só em parte, porque, nos cargos proporcionais, a fração do total chegou mais perto da fração de candidaturas do que esperávamos, e para deputado estadual praticamente a alcançou.

## 5. Quanto arrecada uma candidatura típica

In [10]:
ALVO = ["deputado federal", "vereador"]
d = v[v["cargo"].isin(ALVO)]


def resumo(g):
    pos = g.loc[g["receita_total"] > 0, "receita_total"]
    return pd.Series({
        "candidaturas": len(g),
        "% sem receita": (g["receita_total"] <= 0).mean() * 100,
        "mediana (R$, com receita)": pos.median(),
        "p90 (R$, com receita)": pos.quantile(0.9),
        "mediana financeira (R$)": g.loc[g["receita_financeira"] > 0, "receita_financeira"].median(),
    })


tab = d.groupby(["cargo", "ano", "Gênero"]).apply(resumo, include_groups=False).unstack("Gênero")
tab[("razão F/M", "mediana")] = tab[("mediana (R$, com receita)", "Feminino")] / tab[("mediana (R$, com receita)", "Masculino")]
tab[("razão F/M", "mediana financeira")] = tab[("mediana financeira (R$)", "Feminino")] / tab[("mediana financeira (R$)", "Masculino")]
tab.round(1)

candidaturas           % sem receita            \
Gênero                    Feminino Masculino      Feminino Masculino   
cargo            ano                                                   
deputado federal 2014       1723.0    4145.0          19.6      13.2   
                 2018       2417.0    5212.0           9.2       9.0   
                 2022       3341.0    6114.0           4.6       6.2   
vereador         2016     141495.0  294246.0          10.6       6.3   
                 2020     169271.0  318602.0          19.0      16.0   
                 2024     150780.0  274744.0          19.0      17.4   

                      mediana (R$, com receita)            \
Gênero                                 Feminino Masculino   
cargo            ano                                        
deputado federal 2014                    4632.0   18680.0   
                 2018                   14413.0   18421.0   
                 2022                   82761.3   85000.0   
vereador         2016                     921.8    1748.0   
                 2020                    1373.3    1500.0   
                 2024                    2600.0    2500.0   

                      p90 (R$, com receita)             \
Gênero                             Feminino  Masculino   
cargo            ano                                     
deputado federal 2014              105495.8   949639.8   
                 2018              305139.9   901972.8   
                 2022              823817.0  1352770.0   
vereador         2016                5250.0     8040.2   
                 2020                7968.1     8100.0   
                 2024               16269.3    15659.0   

                      mediana financeira (R$)           razão F/M  \
Gênero                               Feminino Masculino   mediana   
cargo            ano                                                
deputado federal 2014                  7600.0   25829.0       0.2   
                 2018                 17511.0   20958.8       0.8   
                 2022                 80000.0   91000.0       1.0   
vereador         2016                   812.5    1165.0       0.5   
                 2020                  1475.0    1362.7       0.9   
                 2024                  3180.0    2800.0       1.0   

                                          
Gênero                mediana financeira  
cargo            ano                      
deputado federal 2014                0.3  
                 2018                0.8  
                 2022                0.9  
vereador         2016                0.7  
                 2020                1.1  
                 2024                1.1

In [11]:
pos = d[d["receita_total"] > 0].copy()
pos["ano"] = pos["ano"].astype(str)
pos["cargo"] = pos["cargo"].str.title()
pos["receita_total"] = pos["receita_total"].astype(float)

r = tab[("razão F/M", "mediana")]
titulo2 = (f"A candidata mediana arrecadou de {r.min():.0%} a {r.max():.0%} do que arrecadou o candidato mediano"
           +
           "<br><sup>Receita declarada por candidatura válida com receita maior que zero. Escala logarítmica; "
           "valores nominais, compare gêneros dentro de cada ano.</sup>")

fig2 = px.box(
    pos, x="ano", y="receita_total", color="Gênero", facet_col="cargo", points=False, log_y=True,
    color_discrete_map=COR_GENERO,
    category_orders={"cargo": [c.title() for c in ALVO], "Gênero": ["Feminino", "Masculino"]},
    labels={"ano": "Ano da eleição", "receita_total": "Receita da candidatura (R$, escala log)", "Gênero": ""},
    title=titulo2,
)
fig2.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig2.update_xaxes(matches=None)
TICKS = [1e2, 1e3, 1e4, 1e5, 1e6, 1e7]
fig2.update_yaxes(tickvals=TICKS, ticktext=["R$ 100", "R$ 1 mil", "R$ 10 mil", "R$ 100 mil", "R$ 1 mi", "R$ 10 mi"])
fig2.update_layout(boxmode="group", height=520, margin=dict(t=130), title_font_size=16,
                   legend=dict(orientation="h", y=1.12, x=1, xanchor="right"))
fig2.show()

In [12]:
zero = (d.assign(sem_receita=d["receita_total"] <= 0)
          .groupby(["cargo", "ano", "Gênero"])["sem_receita"].agg(n="size", pct="mean").reset_index())
zero["pct"] = zero["pct"] * 100
zero["cargo"] = zero["cargo"].str.title()
zero["ano"] = zero["ano"].astype(int)

fig3 = px.line(
    zero, x="ano", y="pct", color="Gênero", symbol="Gênero", markers=True, facet_col="cargo",
    color_discrete_map=COR_GENERO,
    category_orders={"cargo": [c.title() for c in ALVO], "Gênero": ["Feminino", "Masculino"]},
    hover_data={"n": ":,", "pct": ":.1f", "cargo": False},
    labels={"ano": "Ano da eleição", "pct": "% sem nenhuma receita declarada", "Gênero": "",
            "n": "Candidaturas válidas"},
    title=("Candidaturas válidas sem nenhuma receita declarada, por gênero"
           "<br><sup>Inclui quem não arrecadou e quem não prestou contas; a base não separa os dois casos.</sup>"),
)
fig3.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig3.update_xaxes(matches=None, showticklabels=True, type="category", categoryorder="category ascending")
fig3.update_yaxes(rangemode="tozero", ticksuffix="%")
fig3.update_layout(height=450, margin=dict(t=130), title_font_size=16,
                   legend=dict(orientation="h", y=1.14, x=1, xanchor="right"))
fig3.show()

### Análise: distribuição

No meio da distribuição, a diferença entre candidatas e candidatos praticamente some ao longo do período. Para deputado federal, a candidata mediana arrecadou 4,6 mil reais em 2014, contra 18,7 mil reais do candidato mediano (razão 0,25). A razão sobe para 0,78 em 2018 (14,4 mil contra 18,4 mil reais) e para 0,97 em 2022 (82,8 mil contra 85,0 mil reais). Para vereador, a razão passa de 0,53 em 2016 para 0,92 em 2020 e 1,04 em 2024, quando a candidata mediana arrecadou 2.600 reais, contra 2.500 reais do candidato mediano. Os valores medianos de deputado federal crescem muito entre 2018 e 2022, acompanhando o aumento do Fundo Eleitoral (de 1,8 bilhão para 4,9 bilhões de reais declarados como receita de candidaturas), mas, como os valores são nominais, só comparamos gêneros dentro de cada ano.

No topo, a diferença continua para deputado federal. Entre as candidaturas com receita, os 10% de candidatos que mais arrecadaram declararam ao menos 950 mil reais em 2014, contra 105 mil reais entre as candidatas (razão 0,11). A razão sobe para 0,34 em 2018 e 0,61 em 2022 (824 mil reais contra 1,35 milhão de reais). Como os eleitos estão justamente nesse topo (seção 6), é essa diferença que importa para o resultado. Para vereador, o percentil 90 de mulheres e homens já é praticamente igual em 2020 e 2024.

A fração de candidaturas sem nenhuma receita declarada era maior entre mulheres em 2014 (19,6% contra 13,2% para deputado federal) e em 2016 (10,6% contra 6,3% para vereador). Para deputado federal, as frações se igualam em 2018 e, em 2022, ficam abaixo entre as mulheres (4,6% contra 6,2%). Para vereador, cerca de uma em cada cinco candidaturas femininas não declarou receita em 2020 e 2024 (19,0%), um pouco mais que entre os homens (16,0% e 17,4%). O salto de 2016 para 2020 atinge os dois gêneros, o que sugere uma mudança na forma de prestar contas ou no registro, e não algo específico das mulheres. Essas candidaturas sem receita são o ponto de partida da questão 5.

## 6. Dinheiro e chance de eleição

Para comparar candidaturas com arrecadação parecida, classificamos cada candidatura **dentro da sua disputa**: deputados federais por UF e ano; vereadores por município e ano. As candidaturas com receita viram quintis (Q1 = 20% que menos arrecadaram na disputa, Q5 = 20% que mais arrecadaram) e as sem receita ficam em uma faixa à parte. Usamos as duas últimas eleições de cada cargo, as que já têm a regra dos 30% do dinheiro público.

In [13]:
ULTIMAS = {"deputado federal": [2018, 2022], "vereador": [2020, 2024]}
DISPUTA = {"deputado federal": ["ano", "sigla_uf"], "vereador": ["ano", "id_municipio"]}
FAIXAS = ["Sem receita", "Q1", "Q2", "Q3", "Q4", "Q5"]

partes = []
for cargo, anos in ULTIMAS.items():
    x = v[(v["cargo"] == cargo) & v["ano"].isin(anos)].copy()
    com = x["receita_total"] > 0
    rank = x[com].groupby(DISPUTA[cargo], observed=True)["receita_total"].rank(pct=True)
    x["faixa"] = "Sem receita"
    x.loc[com, "faixa"] = "Q" + np.ceil(rank * 5).clip(1, 5).astype(int).astype(str)
    partes.append(x)
fx = pd.concat(partes)
fx["faixa"] = pd.Categorical(fx["faixa"], categories=FAIXAS, ordered=True)

taxa = (fx.groupby(["cargo", "faixa", "Gênero"], observed=False)["eleito"]
          .agg(n="size", eleitos="sum").reset_index())
taxa["taxa"] = taxa["eleitos"] / taxa["n"] * 100
taxa["pct_do_genero"] = taxa["n"] / taxa.groupby(["cargo", "Gênero"])["n"].transform("sum") * 100
taxa.pivot_table(index=["cargo", "faixa"], columns="Gênero", values=["pct_do_genero", "taxa", "n"], observed=False).round(1)

n           pct_do_genero            \
Gênero                       Feminino Masculino      Feminino Masculino   
cargo            faixa                                                    
deputado federal Sem receita    376.0     846.0           6.5       7.5   
                 Q1            1022.0    2130.0          17.7      18.8   
                 Q2            1144.0    2018.0          19.9      17.8   
                 Q3            1191.0    1989.0          20.7      17.6   
                 Q4            1168.0    2005.0          20.3      17.7   
                 Q5             857.0    2338.0          14.9      20.6   
vereador         Sem receita  60706.0   98738.0          19.0      16.6   
                 Q1           54553.0   89676.0          17.0      15.1   
                 Q2           53756.0  101014.0          16.8      17.0   
                 Q3           50106.0  100360.0          15.7      16.9   
                 Q4           48536.0  101528.0          15.2      17.1   
                 Q5           52394.0  102030.0          16.4      17.2   

                                 taxa            
Gênero                       Feminino Masculino  
cargo            faixa                           
deputado federal Sem receita      0.0       0.0  
                 Q1               0.0       0.0  
                 Q2               0.1       0.1  
                 Q3               0.1       0.7  
                 Q4               0.3       2.1  
                 Q5              18.9      34.2  
vereador         Sem receita      0.9       3.3  
                 Q1               1.5       5.9  
                 Q2               3.1       9.9  
                 Q3               4.8      14.3  
                 Q4               8.2      21.6  
                 Q5              19.9      40.4

In [14]:
t = taxa.copy()
t["cargo"] = t["cargo"].str.title() + " (" + t["cargo"].map(lambda c: " e ".join(map(str, ULTIMAS[c]))) + ")"
t["faixa"] = t["faixa"].astype(str)
ordem_cargo = list(dict.fromkeys(t["cargo"]))

# fração de cada gênero sem receita ou nos dois quintis mais baixos, somando os dois cargos
baixo = taxa[taxa["faixa"].isin(["Sem receita", "Q1", "Q2"])].groupby("Gênero")["n"].sum() / taxa.groupby("Gênero")["n"].sum() * 100
titulo4 = (f"{baixo['Feminino']:.0f}% das candidatas e {baixo['Masculino']:.0f}% dos candidatos estão sem receita "
           "ou entre os 40% que menos arrecadaram na disputa")

fig4 = px.bar(
    t, x="faixa", y="pct_do_genero", color="Gênero", barmode="group", facet_col="cargo",
    color_discrete_map=COR_GENERO,
    category_orders={"faixa": FAIXAS, "Gênero": ["Feminino", "Masculino"], "cargo": ordem_cargo},
    hover_data={"n": ":,", "pct_do_genero": ":.1f", "cargo": False},
    labels={"faixa": "Faixa de arrecadação na disputa", "pct_do_genero": "% das candidaturas do gênero",
            "Gênero": "", "n": "Candidaturas"},
    title=(titulo4 +
           "<br><sup>Distribuição das candidaturas válidas de cada gênero pelas faixas. Q1 a Q5: quintis de receita "
           "dentro da disputa (UF para deputado federal, município para vereador).</sup>"),
)
fig4.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig4.update_xaxes(matches=None)
fig4.update_yaxes(ticksuffix="%")
fig4.update_layout(height=470, bargap=0.25, bargroupgap=0.08, margin=dict(t=130), title_font_size=16,
                   legend=dict(orientation="h", y=1.14, x=1, xanchor="right"))
fig4.show()

In [15]:
fig5 = px.line(
    t, x="faixa", y="taxa", color="Gênero", symbol="Gênero", markers=True, facet_col="cargo",
    color_discrete_map=COR_GENERO,
    category_orders={"faixa": FAIXAS, "Gênero": ["Feminino", "Masculino"], "cargo": ordem_cargo},
    hover_data={"n": ":,", "eleitos": ":,", "taxa": ":.1f", "cargo": False},
    labels={"faixa": "Faixa de arrecadação na disputa", "taxa": "% eleitos na faixa", "Gênero": "",
            "n": "Candidaturas", "eleitos": "Eleitos"},
    title=("Taxa de eleição de candidatas e candidatos por faixa de arrecadação"
           "<br><sup>% de eleitos entre as candidaturas válidas de cada faixa e gênero. "
           "Cada painel tem a sua própria escala vertical.</sup>"),
)
fig5.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig5.update_xaxes(matches=None)
fig5.update_yaxes(matches=None, showticklabels=True, rangemode="tozero", ticksuffix="%")
fig5.update_layout(height=470, margin=dict(t=130), title_font_size=16,
                   legend=dict(orientation="h", y=1.14, x=1, xanchor="right"))
fig5.show()

Para resumir as duas figuras em um número, calculamos uma taxa **reponderada**: a taxa que as candidatas teriam se tivessem a mesma distribuição pelas faixas que os candidatos, mantendo a taxa de eleição delas dentro de cada faixa. A parte da diferença que some com a reponderação é a parte associada a onde as candidatas estão na distribuição de arrecadação; a que sobra aparece mesmo entre candidaturas com arrecadação parecida. É uma decomposição descritiva: não diz que o dinheiro causa a eleição (candidaturas com mais chance também atraem mais dinheiro).

In [16]:
def decompoe(g):
    w = g.pivot(index="faixa", columns="Gênero", values=["taxa", "pct_do_genero"])
    taxa_f = (w[("taxa", "Feminino")] * w[("pct_do_genero", "Feminino")]).sum() / 100
    taxa_m = (w[("taxa", "Masculino")] * w[("pct_do_genero", "Masculino")]).sum() / 100
    taxa_f_rep = (w[("taxa", "Feminino")].fillna(0) * w[("pct_do_genero", "Masculino")]).sum() / 100
    return pd.Series({
        "taxa mulheres (%)": taxa_f,
        "taxa homens (%)": taxa_m,
        "taxa mulheres reponderada (%)": taxa_f_rep,
        "razão F/M": taxa_f / taxa_m,
        "razão F/M reponderada": taxa_f_rep / taxa_m,
        "% da diferença associada à faixa": (taxa_f_rep - taxa_f) / (taxa_m - taxa_f) * 100,
    })


decomp = taxa.groupby("cargo", observed=True).apply(decompoe, include_groups=False)
decomp.round(2)

,taxa mulheres (%),taxa homens (%),taxa mulheres reponderada (%),razão F/M,razão F/M reponderada,% da diferença associada à faixa
cargo,,,,,,
deputado federal,2.92,7.58,3.99,0.39,0.53,23.09
vereador,6.22,16.17,6.56,0.38,0.41,3.43


### Análise: dinheiro e resultado

A Figura 4 mostra que, entre vereadores, a distribuição de candidatas e candidatos pelas faixas é parecida: 19,0% das candidatas e 16,6% dos candidatos não declararam receita, e 16,4% e 17,2% estão no quintil de maior arrecadação da sua disputa. Entre os deputados federais, a diferença está no topo: 14,9% das candidatas estão no quintil mais alto, contra 20,6% dos candidatos, e as candidatas são mais frequentes nos quintis intermediários (Q2 a Q4).

A Figura 5 mostra que a chance de eleição cresce muito com a arrecadação, para os dois gêneros, mas que as candidatas se elegem menos em todas as faixas. Para deputado federal, praticamente ninguém se elege fora do quintil mais alto (no máximo 2,1% no Q4), e dentro dele se elegeram 18,9% das candidatas e 34,2% dos candidatos. Para vereador, a taxa das mulheres fica entre um quarto e metade da dos homens em todas as faixas: 1,5% contra 5,9% no Q1, 4,9% contra 14,2% no Q3 e 19,9% contra 40,4% no Q5. Até entre as candidaturas sem nenhuma receita declarada, 3,3% dos homens se elegeram, contra 0,9% das mulheres.

A taxa reponderada resume as duas figuras. Se as candidatas a deputada federal tivessem a mesma distribuição pelas faixas que os candidatos, a taxa delas passaria de 2,92% para 3,99%, contra 7,58% dos homens: a razão F/M subiria de 0,39 para 0,53, e 23% da diferença desapareceria. Para vereador, a taxa passaria de 6,22% para 6,56%, contra 16,17%, e só 3% da diferença desapareceria. Ou seja, a maior parte da diferença na chance de eleição aparece mesmo entre candidaturas com arrecadação parecida, e a hipótese de que ela diminuiria dentro de cada faixa não se confirma.

Duas ressalvas. Primeiro, os quintis são largos, e no quintil mais alto de deputado federal os homens ocupam a ponta (ver o percentil 90 na seção 5); comparando faixas mais estreitas, a parte associada ao dinheiro provavelmente seria maior que 23%. Segundo, a relação vale nos dois sentidos: quem já tem mandato ou chance real de vitória arrecada mais, e isso pode ajudar a explicar por que candidatos homens, que são a maioria de quem já ocupa os cargos, se elegem mais dentro da mesma faixa.

## 7. Robustez

Repetimos a medida de participação (seção 4) usando só a receita financeira, sem bens e serviços estimáveis. Se as frações mudarem pouco, a conclusão não depende dessa escolha.

In [17]:
rob = v.assign(receita_total=v["receita_financeira"])
comp = participacao(rob, ["ano", "cargo"])[["ano", "cargo", "pct_receita"]].merge(
    part[["ano", "cargo", "pct_receita"]], on=["ano", "cargo"], suffixes=("_financeira", "_total"))
comp["dif_pp"] = comp["pct_receita_financeira"] - comp["pct_receita_total"]
print("maior diferença absoluta (pontos percentuais):", comp["dif_pp"].abs().max().round(2))
comp.round(1)

maior diferença absoluta (pontos percentuais): 5.13


,ano,cargo,pct_receita_financeira,pct_receita_total,dif_pp
0,2014,deputado estadual,12.5,13.1,-0.6
1,2014,deputado federal,9.3,9.8,-0.5
2,2014,governador,5.7,4.6,1.2
3,2014,senador,18.0,16.8,1.2
4,2016,prefeito,11.6,11.8,-0.2
5,2016,vereador,18.4,21.1,-2.7
6,2018,deputado estadual,30.6,30.1,0.5
7,2018,deputado federal,21.9,22.3,-0.4
8,2018,governador,9.9,9.6,0.2
9,2018,senador,16.5,16.3,0.2



De 2018 a 2024, usar só a receita financeira muda a fração das mulheres em no máximo 0,5 ponto percentual, com duas exceções entre vereadores: 2020 (de 37,9% para 32,8%, menos 5,1 pontos) e 2024 (de 25,2% para 24,3%). Em 2020, as receitas estimáveis foram quase metade de tudo o que os candidatos declararam, e uma parte maior delas foi para candidatas. Por isso, para vereador em 2020, a fração do dinheiro em espécie (32,8%) descreve melhor o acesso das mulheres aos recursos do que a fração do total.

*(Conferir depois de rodar de novo: em 2014 e 2016, a coluna de natureza da receita traz a forma de pagamento, e os bens e serviços aparecem como "estimado". A versão anterior do código não reconhecia esse rótulo, por isso a diferença saía zero nesses anos. Com a correção na seção 3.2, as linhas de 2014 e 2016 desta tabela e a "mediana financeira" da seção 5 vão mudar.)*


## 8. Conclusão

Antes da regra de 2018, o dinheiro de campanha ia para as mulheres em uma proporção bem menor que a presença delas nas listas. Em 2014, as candidatas eram 29,4% das candidaturas a deputado federal e receberam 9,8% dos recursos, cerca de um terço do que receberiam se o dinheiro fosse dividido como as candidaturas. Nas eleições seguintes, a distância cai nos cargos proporcionais: em 2022, as candidatas a deputada federal receberam 30,0% dos recursos para 35,3% das candidaturas, e as candidatas a deputada estadual receberam praticamente a mesma fração que tinham das candidaturas. A mediana de arrecadação também se aproxima: em 2022, a candidata a deputada federal mediana arrecadou 97% do que arrecadou o candidato mediano, contra 25% em 2014; entre vereadores, as medianas ficam praticamente iguais em 2020 e 2024. A mudança coincide com o piso de 30% do Fundo Eleitoral, que a partir de 2018 passou a ser a maior fonte de dinheiro das campanhas na maioria das eleições, mas a comparação é descritiva e não isola o efeito da regra. A proibição das doações de empresas, a partir de 2016, e o aumento do próprio fundo ocorreram no mesmo período.

A diferença que sobra está no topo. Entre candidatos a deputado federal, os 10% que mais arrecadaram entre os homens declararam ao menos 1,35 milhão de reais em 2022, contra 824 mil reais entre as mulheres, e só 14,9% das candidatas estão no quintil de maior arrecadação da sua disputa, contra 20,6% dos candidatos. Nos cargos majoritários, a participação das mulheres no dinheiro continua abaixo da participação nas candidaturas para governador. Para prefeito, ela fica acima a partir de 2020, mas as candidatas a prefeita são apenas 13% a 15% do total.

O dinheiro, porém, explica pouco da diferença na chance de eleição vista na questão 2. Em todas as faixas de arrecadação, as candidatas se elegem menos que os candidatos: entre os candidatos a vereador do quintil de maior arrecadação, 19,9% das mulheres e 40,4% dos homens se elegeram. Se as candidatas tivessem a mesma distribuição pelas faixas que os candidatos, a diferença na taxa de eleição diminuiria 23% entre os deputados federais e só 3% entre os vereadores. A hipótese se confirma para o período anterior a 2018 e para o topo da distribuição, mas não para a ideia de que a distribuição do dinheiro explica a maior parte da diferença nos resultados. Outros fatores que os dados desta questão não medem, como a reeleição de quem já tem mandato, a força do partido no município e o tempo de propaganda, devem pesar. Parte deles pode ser examinada na questão 5, a partir das candidaturas com pouca ou nenhuma receita.

## 9. Limitações

- A receita declarada não é o gasto efetivo, e a prestação de contas pode ser retificada depois; usamos a versão disponível na Base dos Dados.
- Candidatura válida sem receita na base pode ter arrecadado e não prestado contas; não é possível separar os dois casos.
- Repasses entre candidaturas (por exemplo, de uma candidata majoritária para a lista) contam como receita de quem recebe, então o total por cargo não é o total de dinheiro novo na eleição. Em 2014, a origem "outros candidatos/comitês" soma 919 milhões de reais.
- Em 2014, 14,8% do valor declarado não foi associado a uma candidatura da análise; quase todo é das campanhas a presidente, que ficaram fora do recorte.
- O piso de 30% vale para o total que cada partido destina às campanhas na eleição, e conta também recursos de chapas majoritárias com vice ou suplente mulher. Como a análise não inclui vices nem suplentes e não está separada por partido, ela não verifica se a regra foi cumprida.
- Valores nominais: comparações em reais só dentro do mesmo ano.
- Os quintis de arrecadação são largos: no quintil mais alto de deputado federal estão campanhas de valores muito diferentes, e os homens se concentram no topo dele. Por isso, a parte da diferença associada ao dinheiro (23%) é provavelmente um valor mínimo; uma divisão mais fina aumentaria esse número.
- A relação entre arrecadação e eleição é descritiva. Candidaturas mais competitivas atraem mais dinheiro, e o dinheiro ajuda a eleger; os dados não separam as duas direções.